# 🛰️ ESG Ratings Are Self-Reported. Satellite Data Isn't.

This notebook cross-references **satellite-measured NO₂ trends** at the primary production facilities of 6 European heavy-industry companies against:
- **E-PRTR self-reported NOₓ emissions** (the official EU industrial emissions registry)
- **5-year total stock return**

The key question: do companies that are *actually* getting cleaner (verified by satellite) outperform those that only *say* they are (self-reported in regulatory filings)?

### What makes this different from a standard ESG screen

We use **E-PRTR facility IDs** — the official EU Environmental Reporting Registry — to pin the query to the exact GPS coordinates of the reporting stack, not a city-level bounding box. This matters: the CAMS 0.1° cell containing a cement kiln is not the same as the 0.1° cell containing the company's administrative office 2 km away.

### Tools used
- **[Jiskta API](https://jiskta.com)** — Copernicus CAMS reanalysis, `aggregate=trend` gives NO₂ slope (µg/m³/year per grid cell). The `return_divergence=True` flag cross-references CAMS against E-PRTR self-reported NOₓ, returning a `direction` label (`consistent` / `divergent`) and a numeric `score`.
- **[yfinance](https://pypi.org/project/yfinance/)** — total stock return 2018–2024

### Results (March 2026, Jiskta API + E-PRTR)

| Company | CAMS NO₂ slope | E-PRTR NOₓ trend | Divergence | Return 2018–2024 |
|---------|---------------|-----------------|------------|------------------|
| RWE (Weisweiler) | −1.095 µg/m³/yr | −642 t/yr | ✅ consistent | +190.8% |
| ArcelorMittal (Ghent) | −1.033 µg/m³/yr | +89 t/yr | ⚠️ divergent | −11.9% |
| BASF (Ludwigshafen) | −0.933 µg/m³/yr | −246 t/yr | ✅ consistent | −25.3% |
| Volkswagen (Wolfsburg) | −0.422 µg/m³/yr | −176 t/yr | ✅ consistent | −0.3% |
| Thyssenkrupp (Bruckhausen) | −0.385 µg/m³/yr | +5 t/yr | ⚠️ divergent | −72.8% |
| **HeidelbergMaterials (Leimen)** | **+0.930 µg/m³/yr** | **−43 t/yr** | **🚩 strongly divergent** | **+7.6%** |

**The actionable signal is divergence** — companies where satellite data and self-reported filings disagree. HeidelbergMaterials reports a 46% NOₓ reduction while the surrounding CAMS grid cell shows rising NO₂.

**Requirements:**
```
pip install jiskta yfinance pandas matplotlib numpy
```

In [ ]:
# Uncomment to install in Colab / fresh environment
# !pip install jiskta yfinance pandas matplotlib numpy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from jiskta import JisktaClient
import yfinance as yf

API_KEY = "sk_live_YOUR_KEY_HERE"  # get one at https://jiskta.com/dashboard
client = JisktaClient(api_key=API_KEY)

print("✓ SDK ready")

## Step 1 — Look up E-PRTR facility IDs

We use `client.facilities()` to find the exact E-PRTR record for each company. This gives us:
- The **verified GPS coordinates** of the reporting stack (not the city)
- The `inspire_hash` — the E-PRTR facility ID we'll pass to `client.query(facility=...)`

Instead of manually guessing bounding boxes, we search by company name and optionally filter by country.

In [ ]:
# Company + ticker pairs — we'll fill in facility data from E-PRTR in the next step
company_defs = [
    {"search": "RWE Power",          "country": "DE", "ticker": "RWE.DE",  "color": "#22c55e", "sector": "Power"},
    {"search": "ArcelorMittal",       "country": "BE", "ticker": "MT",      "color": "#ef4444", "sector": "Steel"},
    {"search": "BASF SE",             "country": "DE", "ticker": "BAS.DE",  "color": "#f97316", "sector": "Chemicals"},
    {"search": "Volkswagen",          "country": "DE", "ticker": "VOW3.DE", "color": "#3b82f6", "sector": "Automotive"},
    {"search": "Thyssenkrupp Steel",  "country": "DE", "ticker": "TKA.DE",  "color": "#ef4444", "sector": "Steel"},
    {"search": "Heidelberg",          "country": "DE", "ticker": "HEI.DE",  "color": "#a855f7", "sector": "Cement"},
]

# Known verified facility IDs from E-PRTR (pre-verified; keeps notebook reproducible
# even if the free-text search order changes between API versions)
VERIFIED_IDS = {
    "RWE.DE":  12394198829192159256,  # RWE Power AG — Weisweiler coal plant (lat 50.833, lon 6.315)
    "MT":       1986304935103026946,  # ArcelorMittal Belgium — Ghent (lat 51.177, lon 3.810)
    "BAS.DE":  12626246128710077482,  # BASF SE — Ludwigshafen (lat 49.517, lon 8.423)
    "VOW3.DE": 13408216860473500983,  # Volkswagen AG — Wolfsburg (lat 52.430, lon 10.768)
    "TKA.DE":  11332985180430661771,  # Thyssenkrupp Steel — Bruckhausen (lat 51.494, lon 6.738)
    "HEI.DE":   3687905997148585988,  # HeidelbergCement — Leimen (lat 49.358, lon 8.686)
}

companies = []
print("Looking up E-PRTR facility records...\n")

for cd in company_defs:
    fid = VERIFIED_IDS[cd["ticker"]]

    # Direct ID lookup — most reliable; avoids text-search ambiguity
    result = client.facilities(id=fid, emissions=True)
    fac = result if isinstance(result, dict) else (result[0] if result else None)

    if fac is None:
        print(f"  ⚠ Could not find facility for {cd['search']} — trying text search")
        candidates = client.facilities(q=cd["search"], country=cd["country"])
        if not candidates:
            print(f"  ✗ No results — skipping {cd['search']}")
            continue
        fac = candidates[0] if isinstance(candidates, list) else candidates

    record = {
        **cd,
        "name":         fac["name"],
        "inspire_hash": fac["inspire_hash"],
        "lat":          fac["lat"],
        "lon":          fac["lon"],
        "country":      fac["country"],
    }
    companies.append(record)
    print(f"  ✓ {fac['name'][:55]:<55}  id={fac['inspire_hash']}  ({fac['lat']:.3f}°N, {fac['lon']:.3f}°E)")

print(f"\n{len(companies)} / {len(company_defs)} facilities found")

## Step 2 — Query CAMS NO₂ trend + E-PRTR divergence

`aggregate="trend"` runs OLS linear regression on 72 months of Copernicus CAMS reanalysis (Jan 2018 – Dec 2023). The result is a `slope` in µg/m³/year for each CAMS 0.1° grid cell.

With `return_divergence=True`, the API also returns the divergence assessment:
- Fetches this facility's E-PRTR annual NOₓ reports for the same period
- Computes the E-PRTR trend slope (tonnes/year)
- Labels the pair as `consistent`, `divergent`, or `strongly_divergent`
- Returns a numeric `score` (−1 = perfect inverse, 0 = independent, +1 = perfect agreement)

In [ ]:
print("Querying CAMS NO₂ trends + E-PRTR divergence (2018–2023)...\n")

for c in companies:
    df_trend, divergence = client.query(
        facility=c["inspire_hash"],   # pins query to verified E-PRTR GPS
        start="2018-01",
        end="2023-12",
        variables=["no2"],
        aggregate="trend",
        return_divergence=True,
    )

    c["no2_slope"]       = df_trend["slope"].iloc[0]
    c["no2_r2"]          = df_trend["r2"].iloc[0]
    c["divergence"]      = divergence  # full dict
    c["div_direction"]   = divergence.get("direction", "unknown") if divergence else "n/a"
    c["div_score"]       = divergence.get("score", None)         if divergence else None
    c["eprtr_nox_slope"] = divergence.get("eprtr_nox_slope", None) if divergence else None

    # Human-readable divergence flag
    flag = {"consistent": "✅", "divergent": "⚠️", "strongly_divergent": "🚩"}.get(c["div_direction"], "❔")
    print(
        f"  {c['ticker']:<9}"
        f"  CAMS={c['no2_slope']:+.3f} µg/m³/yr"
        f"  E-PRTR NOₓ={c['eprtr_nox_slope']:+.0f} t/yr" if c["eprtr_nox_slope"] else "",
        f"  {flag} {c['div_direction']}"
    )

print("\nDone.")

## Step 3 — Fetch 5-year stock returns via yfinance

In [ ]:
START_DATE = "2018-01-01"
END_DATE   = "2024-01-01"

print(f"Fetching stock returns {START_DATE} → {END_DATE}...\n")

for c in companies:
    try:
        hist = yf.Ticker(c["ticker"]).history(start=START_DATE, end=END_DATE, auto_adjust=True)
        if len(hist) < 50:
            print(f"  {c['ticker']:<12} ⚠ insufficient data ({len(hist)} days)")
            c["total_return_pct"] = float("nan")
            continue
        ret = (hist["Close"].iloc[-1] / hist["Close"].iloc[0] - 1) * 100
        c["total_return_pct"] = round(ret, 1)
        print(f"  {c['ticker']:<12} {ret:+.1f}%  ({len(hist)} trading days)")
    except Exception as e:
        print(f"  {c['ticker']:<12} ✗ {e}")
        c["total_return_pct"] = float("nan")

## Step 4 — Build the summary DataFrame

In [ ]:
df = pd.DataFrame(companies)[[
    "ticker", "name", "sector",
    "no2_slope", "no2_r2",
    "eprtr_nox_slope", "div_direction", "div_score",
    "total_return_pct",
]].copy()

# Short display name for charts
df["short_name"] = df["name"].str.split(r"[/,]").str[0].str.strip()

# Improvement rate: positive = getting cleaner by satellite
df["cams_improvement"] = -df["no2_slope"]

# Sort by CAMS improvement descending
df = df.sort_values("cams_improvement", ascending=False).reset_index(drop=True)

pd.set_option("display.max_colwidth", 55)
print(df[[
    "ticker", "no2_slope", "eprtr_nox_slope", "div_direction", "total_return_pct"
]].to_string(index=False))

## Step 5 — Visualise

### Chart A — Who is actually reducing NO₂ at the source? (CAMS satellite)

In [ ]:
color_map = {c["ticker"]: c["color"] for c in companies}
div_colors = {"consistent": "#22c55e", "divergent": "#f59e0b", "strongly_divergent": "#ef4444"}

fig, ax = plt.subplots(figsize=(11, 5))

bar_colors = [
    div_colors.get(row["div_direction"], "#94a3b8")
    for _, row in df.iterrows()
]

bars = ax.barh(
    df["short_name"],
    df["cams_improvement"],
    color=bar_colors,
    height=0.55,
    edgecolor="white",
    linewidth=0.5,
)

for bar, (_, row) in zip(bars, df.iterrows()):
    label = f"{row['cams_improvement']:.3f} µg/m³/yr"
    x = bar.get_width() + 0.02 if bar.get_width() >= 0 else bar.get_width() - 0.02
    ha = "left" if bar.get_width() >= 0 else "right"
    ax.text(x, bar.get_y() + bar.get_height() / 2, label, va="center", ha=ha, fontsize=8.5)

ax.axvline(0, color="#94a3b8", linewidth=0.8)
ax.set_xlabel("Annual NO₂ change at facility (µg/m³ per year) — negative = improving")
ax.set_title(
    "CAMS Satellite NO₂ Trend at Primary Production Facility — 2018 to 2023\n"
    "Colour = divergence vs E-PRTR self-reported NOₓ: green=consistent, amber=divergent, red=strongly divergent",
    fontsize=10
)
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)

legend_patches = [
    mpatches.Patch(color="#22c55e", label="Consistent (satellite ≈ E-PRTR)"),
    mpatches.Patch(color="#f59e0b", label="Divergent (satellite improves, E-PRTR flat)"),
    mpatches.Patch(color="#ef4444", label="Strongly divergent (opposite signals)"),
]
ax.legend(handles=legend_patches, fontsize=8, loc="lower right")
fig.tight_layout()
plt.savefig("chart_a_cams_trend.png", dpi=150, bbox_inches="tight")
plt.show()

### Chart B — CAMS satellite vs E-PRTR self-reported: where do they disagree?

In [ ]:
df_div = df.dropna(subset=["eprtr_nox_slope"]).copy()

# Normalise E-PRTR slope to 0–1 range for visual comparison
# (units differ: µg/m³/yr vs tonnes/yr — we only care about direction + relative magnitude)
eprtr_norm = df_div["eprtr_nox_slope"] / df_div["eprtr_nox_slope"].abs().max()
cams_norm  = -df_div["no2_slope"]  / df_div["no2_slope"].abs().max()  # flip sign: positive = improving

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(df_div))
width = 0.38

bars1 = ax.bar(x - width/2, cams_norm,  width, label="CAMS satellite (normalised)",  color="#3b82f6", alpha=0.85)
bars2 = ax.bar(x + width/2, -eprtr_norm, width, label="E-PRTR self-reported (normalised, negative=improving)",
               color="#64748b", alpha=0.65)

ax.axhline(0, color="#94a3b8", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(df_div["short_name"].str.split().str[0], fontsize=9)
ax.set_ylabel("Normalised improvement score (positive = getting cleaner)")
ax.set_title(
    "Satellite vs Self-Reported: Do They Tell the Same Story?\n"
    "Blue bars (CAMS) vs grey bars (E-PRTR): divergence = width gap on opposite sides",
    fontsize=10
)
ax.legend(fontsize=8)
ax.grid(axis="y", alpha=0.3)

# Annotate divergence labels
for i, (_, row) in enumerate(df_div.iterrows()):
    flag = {"consistent": "✅", "divergent": "⚠️", "strongly_divergent": "🚩"}.get(row["div_direction"], "")
    ax.text(i, ax.get_ylim()[0] - 0.08, flag, ha="center", fontsize=12)

fig.tight_layout()
plt.savefig("chart_b_divergence.png", dpi=150, bbox_inches="tight")
plt.show()

### Chart C — Does satellite-measured improvement predict stock returns?

In [ ]:
df_plot = df.dropna(subset=["total_return_pct"]).copy()

fig, ax = plt.subplots(figsize=(9, 6))

for _, row in df_plot.iterrows():
    clr = color_map.get(row["ticker"], "#94a3b8")
    ax.scatter(row["cams_improvement"], row["total_return_pct"],
               color=clr, s=160, zorder=5, edgecolors="white", linewidth=1.2)
    offset = (8, 6) if row["ticker"] != "TKA.DE" else (8, -16)
    ax.annotate(
        row["short_name"].split()[0],
        (row["cams_improvement"], row["total_return_pct"]),
        textcoords="offset points", xytext=offset, fontsize=8.5,
    )

# OLS trend line — note Thyssenkrupp is a restructuring outlier
m, b = np.polyfit(df_plot["cams_improvement"], df_plot["total_return_pct"], 1)
x_line = np.linspace(df_plot["cams_improvement"].min() - 0.1,
                     df_plot["cams_improvement"].max() + 0.1, 100)
ax.plot(x_line, m * x_line + b, color="#94a3b8", linewidth=1.5,
        linestyle="--", alpha=0.8, label=f"OLS fit (all): slope={m:.0f}% per µg/m³/yr")

r2_all = np.corrcoef(df_plot["cams_improvement"], df_plot["total_return_pct"])[0, 1] ** 2
ax.text(0.05, 0.95, f"R² = {r2_all:.2f}  (N=6)",
        transform=ax.transAxes, fontsize=9, color="#64748b", va="top")

ax.axhline(0, color="#cbd5e1", linewidth=0.8)
ax.axvline(0, color="#cbd5e1", linewidth=0.8)
ax.set_xlabel(
    "CAMS NO₂ improvement rate at facility (µg/m³ per year)\n← Less improvement     More improvement →",
    fontsize=10
)
ax.set_ylabel("Total stock return, Jan 2018 → Jan 2024 (%)", fontsize=10)
ax.set_title(
    "Do Companies Getting Cleaner (by Satellite) Outperform?\n"
    "6 European heavy-industry companies — Jiskta API + yfinance",
    fontsize=11, fontweight="bold"
)
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.savefig("chart_c_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"OLS slope: {m:.1f}% return per µg/m³/yr improvement")
print(f"R² (all companies): {r2_all:.2f}")
print("\nInterpretation: R² ≈ 0 means markets are NOT pricing emission trajectories.")
print("The actionable signal is divergence (satellite ≠ self-reported), not correlation.")

## Discussion

### Why R² ≈ 0 is the most important result

With only 6 data points, the scatter is expected to be wide. But the near-zero R² matches the broader literature: markets are not systematically pricing facility-level emission trajectories. This is the opportunity. If satellite data is a leading indicator of Scope 1 capex spending (it takes 3–5 years for new scrubbers or fuel switches to show up in CAMS at statistically significant levels), then divergence between satellite and self-reported signals represents an information edge.

### Three types of company in this dataset

**✅ Consistent improvers (CAMS ≈ E-PRTR):**
- **RWE Weisweiler** (CAMS −1.095 µg/m³/yr, E-PRTR −642 t/yr NOₓ): coal plant closures visible from orbit. Best stock in the cohort (+190.8%).
- **BASF Ludwigshafen** (CAMS −0.933, E-PRTR −246 t/yr): consistent improvement, weak returns (−25.3%) driven by 2022 gas crisis, not the emission trajectory.
- **VW Wolfsburg** (CAMS −0.422, E-PRTR −176 t/yr): paint shop and onsite power upgrades visible. Flat stock (−0.3%) — EV transition uncertainty dominates.

**⚠️ Divergent: satellite improves, E-PRTR flat:**
- **ArcelorMittal Ghent** (CAMS −1.033, E-PRTR +89 t/yr): satellite sees improvement, but the facility itself reports flat/rising NOₓ. The CAMS cell likely captures nearby industrial closures — ArcelorMittal is getting credit for its neighbours' decarbonisation.
- **Thyssenkrupp Bruckhausen** (CAMS −0.385, E-PRTR +5 t/yr): similar story, but at a much larger integrated steel complex. The divergence score flags that the self-reported trajectory and satellite signal are pointing in different directions.

**🚩 Strongly divergent: opposite signals:**
- **HeidelbergMaterials Leimen** (CAMS +0.930, E-PRTR −43 t/yr): this is the most interesting case. E-PRTR shows a **46% NOₓ reduction** (262 → 142 t over 2018–2023), consistent with their CCUS pilot and low-carbon cement narrative. But CAMS shows NO₂ *rising* at the same grid cell. Possible explanations: (1) the stack-level NOₓ reduction is genuine but additional logistics/transport activity near the plant is increasing ambient NO₂; (2) the E-PRTR number captures only the kiln stack while CAMS captures the full 0.1° cell including quarry vehicles, delivery trucks, and new nearby industrial activity. This divergence warrants investigation before using the E-PRTR figure in an ESG score.

### Extensions

```python
# 1. Add more companies — the API handles hundreds of facility queries cheaply
# 2. Use aggregate="monthly" and correlate NO₂ momentum with earnings surprise windows
# 3. Extend to Asian steel / Chinese power using CAMS Global (0.75°, 2020–present)
# 4. Layer in ERA5 wind speed to control for meteorological dispersion effects
# 5. Use query_with_mask(mask_type="urban") to exclude urban background from facility cell
```